# AuraVoice: AI Speech Authenticity Classification & Training Pipeline
## Deep Learning Classifier using MFCC Feature Extraction and 2D CNN Architecture

This notebook outlines the training pipeline to build a Convolutional Neural Network (CNN) in PyTorch to classify audio recordings as **Genuine (Human)** or **Deepfake (AI-synthesized)**.

### Key Improvements:
- **Automatic Dataset Resolution**: Locates the dataset root directory under `/kaggle/input` dynamically.
- **In-Memory Caching**: Pre-extracts acoustic features into RAM before the training loop, accelerating epochs from hours to minutes.

In [ ]:
# Install essential libraries (silent)
!pip install -q librosa scikit-learn pandas numpy matplotlib seaborn tqdm torch torchaudio

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Parameters & Directory Settings

In [ ]:
# Acoustic parameters
SAMPLE_RATE = 16000
N_MFCC = 40
MAX_LEN = 150  # Target frame width (approx 3.0 seconds)

# Model training configuration
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 0.001
MAX_SAMPLES_PER_CLASS = 2000  # Cap class size to manage Kaggle memory limits

def discover_dataset_path():
    if not os.path.exists('/kaggle/input'):
        return 'dataset/LA_norm/train'
        
    found_path = None
    # Search input folder recursively for directories containing 'real' and 'fake'
    for root, subdirs, _ in os.walk('/kaggle/input'):
        lower_subs = [s.lower() for s in subdirs]
        if ('real' in lower_subs and 'fake' in lower_subs) or ('genuine' in lower_subs and 'spoof' in lower_subs):
            if 'train' in root.lower() or 'training' in root.lower():
                return root
            found_path = root
            
    if found_path:
        return found_path
    return '/kaggle/input/the-fake-or-real-dataset/for-norm/for-norm/training'

DATA_DIR = discover_dataset_path()
print(f"Resolved dataset path: {DATA_DIR}")

## 2. Feature Extraction & Caching

In [ ]:
def extract_features(file_path):
    """
    Load audio and compute temporal MFCC features.
    """
    try:
        vocal_sig, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        mfccs = librosa.feature.mfcc(y=vocal_sig, sr=sr, n_mfcc=N_MFCC)
        
        # Pad/truncate
        _, cols = mfccs.shape
        if cols < MAX_LEN:
            pad_width = MAX_LEN - cols
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :MAX_LEN]
            
        return mfccs
    except Exception:
        return None

def cache_dataset_features(file_list):
    """
    Pre-extract features for all clips into a single array to optimize speed.
    """
    features_buffer = []
    for file in tqdm(file_list, desc="Pre-processing Audio"):
        mfcc_feat = extract_features(file)
        if mfcc_feat is None:
            mfcc_feat = np.zeros((N_MFCC, MAX_LEN))
        features_buffer.append(mfcc_feat)
    return np.array(features_buffer)

class AudioDataset(Dataset):
    def __init__(self, pre_extracted_features, labels):
        self.features = pre_extracted_features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feat = self.features[idx][np.newaxis, ...]
        return torch.tensor(feat, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.float32)

## 3. Network Architecture (AudioCNN)

In [ ]:
class AudioCNN(nn.Module):
    """
    2D CNN Classifier mapping to original weights specifications.
    """
    def __init__(self):
        super(AudioCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.flat_size = 64 * (N_MFCC // 8) * (MAX_LEN // 8)
        self.fc1 = nn.Linear(self.flat_size, 128)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = x.view(-1, self.flat_size)
        x = self.dropout(self.relu4(self.fc1(x)))
        x = self.fc2(x)
        return self.sigmoid(x).squeeze()

## 4. Evaluation EER Utility

In [ ]:
def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
    return fpr[eer_idx]

## 5. Load and Pre-Extract Data

In [ ]:
genuine_dirs = [os.path.join(DATA_DIR, x) for x in ['genuine', 'real', 'train/genuine', 'train/real']]
spoof_dirs = [os.path.join(DATA_DIR, x) for x in ['spoof', 'fake', 'train/spoof', 'train/fake']]

genuine_dir = next((d for d in genuine_dirs if os.path.exists(d)), None)
spoof_dir = next((d for d in spoof_dirs if os.path.exists(d)), None)

if genuine_dir and spoof_dir:
    genuine_files = glob.glob(os.path.join(genuine_dir, '*.wav'))[:MAX_SAMPLES_PER_CLASS]
    spoof_files = glob.glob(os.path.join(spoof_dir, '*.wav'))[:MAX_SAMPLES_PER_CLASS]
    
    files = genuine_files + spoof_files
    labels = [1] * len(genuine_files) + [0] * len(spoof_files)
    print(f"Found {len(genuine_files)} Genuine files and {len(spoof_files)} Spoof files.")
else:
    # local/mock fallback
    all_wavs = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)
    files = []
    labels = []
    for f in all_wavs:
        files.append(f)
        if 'spoof' in f.lower() or 'fake' in f.lower():
            labels.append(0)
        else:
            labels.append(1)
    print(f"Fallback loaded {len(files)} total files.")

In [ ]:
if len(files) > 0:
    X_train, X_val, y_train, y_val = train_test_split(
        files, labels, test_size=0.2, random_state=42, stratify=labels if len(np.unique(labels)) > 1 else None
    )
    
    print("\n--- Starting Audio Pre-Extraction ---")
    X_train_features = cache_dataset_features(X_train)
    X_val_features = cache_dataset_features(X_val)
    
    train_loader = DataLoader(AudioDataset(X_train_features, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(AudioDataset(X_val_features, y_val), batch_size=BATCH_SIZE, shuffle=False)
    
    print("Features loaded into RAM.")
else:
    print("Error: No audio files loaded.")

## 6. Training Execution

In [ ]:
if len(files) > 0:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Accelerator device: {device}")
    
    model = AudioCNN().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_f1 = 0.0
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
            
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)
        
        model.eval()
        val_loss = 0.0
        all_targets, all_outputs = [], []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
                
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                all_targets.extend(targets.cpu().numpy())
                all_outputs.extend(outputs.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        
        predictions = (np.array(all_outputs) > 0.5).astype(int)
        acc = accuracy_score(all_targets, predictions)
        f1 = f1_score(all_targets, predictions, zero_division=0)
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
        
        if f1 >= best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), 'deepfake_audio_model.pth')
else:
    print("No training data, skipping.")

## 7. Performance Reports

In [ ]:
if len(files) > 0:
    model.load_state_dict(torch.load('deepfake_audio_model.pth', map_location=device))
    model.eval()
    
    all_targets, all_outputs = [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
            all_targets.extend(targets.numpy())
            all_outputs.extend(outputs.cpu().numpy())
            
    y_true = np.array(all_targets)
    y_scores = np.array(all_outputs)
    predictions = (y_scores > 0.5).astype(int)
    
    print("\n================ PERFORMANCE REPORT ================")
    print("Confusion Matrix:\n", confusion_matrix(y_true, predictions))
    print(f"Overall Accuracy: {accuracy_score(y_true, predictions):.4f}")
    print(f"F1 Score:         {f1_score(y_true, predictions, zero_division=0):.4f}")
    print(f"Equal Error Rate (EER): {compute_eer(y_true, y_scores):.4f}")
    print("====================================================\n")
else:
    print("No evaluation data available.")